# Dataset

Bloc C — construction du dataset de dialogues (JSONL) à partir des fiches concept (T4.1).

Les cellules s'exécutent dans l'ordre. Le dossier `json` des fiches doit être à côté du notebook.

In [ ]:
import json
from pathlib import Path

# Chemin relatif : le script se lance depuis la racine du dépôt.
FICHES_FOLDER = "json"
REQUIRED_KEYS = ["id", "concept", "explication", "questions"]
VERDICTS = ["PARFAIT", "PARTIEL", "INCORRECT"]
# Pas de nom versionné (dataset_v0.1.jsonl) tant que les tours sont des textes d'essai.
OUTPUT_PATH = "dataset_placeholder.jsonl"
# Le LLM de Colab n'a pas de réglage de température : on réessaie quand sa sortie est inutilisable.
MAX_ATTEMPTS = 3
# Règles de verdict de la US2 (backlog), rappelées au LLM qui écrit les réponses d'apprenant.
VERDICT_RULES = {
    "PARFAIT": "tous les points clés présents, aucune erreur",
    "PARTIEL": "compréhension globalement correcte, mais incomplète ou avec une imprécision mineure",
    "INCORRECT": "aucun point clé correct, ou une erreur majeure qui contredit un point clé",
}

# Texte provisoire : à remplacer par le contenu de prompt_v0.1.txt (bloc A) dès qu'il existe.
SYSTEM_PROMPT = (
    "Tu es un tuteur. Tu évalues la réponse de l'apprenant à la question ci-dessous "
    "à partir des points clés, sans donner la réponse.\n"
    "Ta première ligne est toujours [VERDICT: PARFAIT], [VERDICT: PARTIEL] ou [VERDICT: INCORRECT], "
    "suivie d'un feedback."
)

In [ ]:
def load_fiches(folder: str = FICHES_FOLDER) -> list:
    """Charge toutes les fiches concept d'un dossier.

    Paramètres :
        folder : chemin du dossier qui contient les fichiers .json.

    Retour :
        liste de fiches (dictionnaires), triée par "id".

    Exemple :
        load_fiches("json")[0]["id"]  # "bateau-de-thesee"
    """
    fiches = []
    for path in Path(folder).glob("*.json"):
        fiche = json.loads(path.read_text(encoding="utf-8"))
        for key in REQUIRED_KEYS:
            # une fiche incomplète produirait des dialogues faux sans erreur visible
            if key not in fiche:
                raise ValueError(f"{path.name} : clé manquante « {key} »")
        fiches.append(fiche)
    if not fiches:
        raise ValueError(f"aucune fiche trouvée dans le dossier « {folder} »")
    # ordre fixe : le dataset généré est le même d'une exécution à l'autre
    return sorted(fiches, key=lambda fiche: fiche["id"])


# load_fiches : ordre stable, et chaque question appartient bien à sa fiche.
fiches = load_fiches()
assert [fiche["id"] for fiche in fiches] == sorted(fiche["id"] for fiche in fiches)
assert all(q["id"].startswith(fiche["id"]) for fiche in fiches for q in fiche["questions"])
print(f"load_fiches : {len(fiches)} fiches, {sum(len(fiche['questions']) for fiche in fiches)} questions")

In [ ]:
def build_system_message(fiche: dict, question: dict, prompt: str = SYSTEM_PROMPT) -> str:
    """Assemble le message système pour une question d'une fiche.

    Paramètres :
        fiche : fiche concept chargée par load_fiches.
        question : une des questions de cette fiche.
        prompt : consignes données à l'agent.

    Retour :
        texte du message système : consignes, concept, explication lue, question, points clés.

    Exemple :
        build_system_message(fiches[0], fiches[0]["questions"][0]).startswith("Tu es un tuteur")  # True
    """
    # l'apprenant n'a lu que les sections placées avant la question : on limite aussi la longueur
    sections_read = fiche["explication"][: question["apres_section"]]
    explication = "\n\n".join(section["contenu"] for section in sections_read)
    points_cles = "\n".join(f"- {point}" for point in question["points_cles"])
    return (
        f"{prompt}\n\n"
        f"Concept : {fiche['concept']}\n\n"
        f"Explication :\n{explication}\n\n"
        f"Question : {question['enonce']}\n\n"
        f"Points clés :\n{points_cles}"
    )


# build_system_message : la question, ses points clés et seulement les sections déjà lues.
first_question = fiches[0]["questions"][0]
message = build_system_message(fiches[0], first_question)
sections_count = first_question["apres_section"]
assert message.startswith(SYSTEM_PROMPT)
assert first_question["enonce"] in message
assert all(point in message for point in first_question["points_cles"])
assert fiches[0]["explication"][sections_count - 1]["contenu"] in message
assert fiches[0]["explication"][sections_count]["contenu"] not in message
print(f"build_system_message : {len(message)} caractères pour {first_question['id']}")

In [ ]:
def build_dialogue(fiche: dict, question: dict, turns: list) -> dict:
    """Assemble un dialogue du dataset pour une question.

    Paramètres :
        fiche : fiche concept chargée par load_fiches.
        question : une des questions de cette fiche.
        turns : liste de tours {"user", "verdict", "feedback"}, dans l'ordre du dialogue.

    Retour :
        {"messages": [system, user, assistant, ...]} (format T4.1, réponses au format du contrat 2).

    Exemple :
        build_dialogue(fiche, question, [{"user": "Je change.", "verdict": "PARTIEL", "feedback": "Pourquoi ?"}])
    """
    messages = [{"role": "system", "content": build_system_message(fiche, question)}]
    for turn in turns:
        # un verdict hors liste donnerait un exemple d'entraînement contraire au contrat 2
        if turn["verdict"] not in VERDICTS:
            raise ValueError(f"verdict inconnu : {turn['verdict']}")
        messages.append({"role": "user", "content": turn["user"]})
        messages.append({"role": "assistant", "content": f"[VERDICT: {turn['verdict']}]\n{turn['feedback']}"})
    return {"messages": messages}


# build_dialogue : 2 tours donnent system + 2 × (user, assistant), au format du contrat 2.
turns = [
    {"user": "première réponse d'essai", "verdict": "INCORRECT", "feedback": "feedback d'essai 1"},
    {"user": "deuxième réponse d'essai", "verdict": "PARFAIT", "feedback": "feedback d'essai 2"},
]
dialogue = build_dialogue(fiches[0], first_question, turns)
roles = [message["role"] for message in dialogue["messages"]]
assert roles == ["system", "user", "assistant", "user", "assistant"]
assert dialogue["messages"][2]["content"] == "[VERDICT: INCORRECT]\nfeedback d'essai 1"
try:
    build_dialogue(fiches[0], first_question, [{"user": "x", "verdict": "BRAVO", "feedback": "y"}])
    assert False, "un verdict inconnu doit lever une erreur"
except ValueError as error:
    print("build_dialogue : erreur attendue →", error)

In [ ]:
def build_dataset(fiches: list, seed: list) -> list:
    """Construit tous les dialogues décrits dans la graine.

    Paramètres :
        fiches : fiches chargées par load_fiches.
        seed : liste d'entrées {"question_id", "turns"} ; "turns" suit le format de build_dialogue.

    Retour :
        liste de dialogues {"messages": [...]}, dans l'ordre de la graine.

    Exemple :
        build_dataset(fiches, [{"question_id": "monty-hall_q2", "turns": turns}])
    """
    # index par identifiant de question : chaque entrée de la graine retrouve sa fiche directement
    questions_by_id = {}
    for fiche in fiches:
        for question in fiche["questions"]:
            questions_by_id[question["id"]] = (fiche, question)

    dialogues = []
    for entry in seed:
        # une faute de frappe dans la graine doit s'arrêter ici, pas produire un dialogue sur une autre question
        if entry["question_id"] not in questions_by_id:
            raise ValueError(f"question inconnue : {entry['question_id']}")
        fiche, question = questions_by_id[entry["question_id"]]
        dialogues.append(build_dialogue(fiche, question, entry["turns"]))
    return dialogues


# build_dataset : un dialogue par entrée de la graine, sur la bonne question, dans le même ordre.
last_question = fiches[-1]["questions"][-1]
seed = [
    {"question_id": last_question["id"], "turns": turns},
    {"question_id": first_question["id"], "turns": turns[:1]},
]
dialogues = build_dataset(fiches, seed)
assert last_question["enonce"] in dialogues[0]["messages"][0]["content"]
assert [len(dialogue["messages"]) for dialogue in dialogues] == [5, 3]
try:
    build_dataset(fiches, [{"question_id": "question_absente", "turns": turns}])
    assert False, "une question inconnue doit lever une erreur"
except ValueError as error:
    print("build_dataset : erreur attendue →", error)

In [ ]:
def write_jsonl(dialogues: list, path: str) -> int:
    """Écrit les dialogues dans un fichier JSONL, un dialogue par ligne.

    Paramètres :
        dialogues : liste de dialogues {"messages": [...]}.
        path : chemin du fichier à écrire (exemple : "dataset_v0.1.jsonl").

    Retour :
        nombre de lignes écrites.

    Exemple :
        write_jsonl(dialogues, "dataset_v0.1.jsonl")  # 30
    """
    with open(path, "w", encoding="utf-8") as file:
        for dialogue in dialogues:
            # ensure_ascii=False garde les accents lisibles pour la relecture à la main
            file.write(json.dumps(dialogue, ensure_ascii=False) + "\n")
    return len(dialogues)


# write_jsonl : le fichier relu ligne par ligne redonne exactement les dialogues écrits.
test_path = "dataset_test.jsonl"
lines_count = write_jsonl(dialogues, test_path)
with open(test_path, encoding="utf-8") as file:
    reloaded = [json.loads(line) for line in file]
Path(test_path).unlink()  # fichier d'essai : on ne le laisse pas à côté des vrais datasets
assert lines_count == 2 and reloaded == dialogues
print(f"write_jsonl : {lines_count} lignes écrites puis relues à l'identique")

In [ ]:
def generate_turn(fiche: dict, question: dict, verdict: str, llm) -> dict:
    """Demande au LLM une réponse d'apprenant qui mérite le verdict donné, et le feedback du tuteur.

    Paramètres :
        fiche : fiche concept chargée par load_fiches.
        question : une des questions de cette fiche.
        verdict : "PARFAIT", "PARTIEL" ou "INCORRECT".
        llm : fonction qui reçoit un prompt et renvoie un texte (sur Colab : ai.generate_text).

    Retour :
        tour {"user", "verdict", "feedback"}, au format attendu par build_dialogue.

    Exemple :
        generate_turn(fiche, question, "PARTIEL", ai.generate_text)["verdict"]  # "PARTIEL"
    """
    if verdict not in VERDICTS:
        raise ValueError(f"verdict inconnu : {verdict}")
    # mêmes sections que dans le message système : le LLM connaît la mise en scène vue par l'apprenant
    sections_read = fiche["explication"][: question["apres_section"]]
    explication = "\n\n".join(section["contenu"] for section in sections_read)
    points_cles = "\n".join(f"- {point}" for point in question["points_cles"])
    prompt = (
        "Tu aides à créer des exemples pour entraîner un tuteur.\n\n"
        f"Concept : {fiche['concept']}\n\n"
        f"Texte lu par l'apprenant :\n{explication}\n\n"
        f"Question posée à l'apprenant : {question['enonce']}\n\n"
        f"Points clés attendus :\n{points_cles}\n\n"
        f"Écris la réponse d'un apprenant qui mérite le verdict {verdict} ({VERDICT_RULES[verdict]}), "
        "puis le feedback du tuteur : il dit ce qui est compris et ce qui manque, "
        "sans donner la réponse complète.\n\n"
        'Réponds uniquement avec un objet JSON de la forme {"user": "...", "feedback": "..."}.'
    )
    for _ in range(MAX_ATTEMPTS):
        raw = llm(prompt)
        # les LLM entourent souvent le JSON de texte ou de ```json : on garde ce qui est entre accolades
        json_text = raw[raw.find("{") : raw.rfind("}") + 1]
        try:
            data = json.loads(json_text)
            user, feedback = data["user"].strip(), data["feedback"].strip()
        except (ValueError, KeyError, TypeError, AttributeError):
            continue  # sortie inutilisable : on redemande
        # build_dialogue ajoute lui-même la ligne [VERDICT: X] : le feedback ne doit pas la contenir
        if user and feedback and "[VERDICT" not in feedback:
            return {"user": user, "verdict": verdict, "feedback": feedback}
    raise ValueError(f"{question['id']} ({verdict}) : aucune sortie valide après {MAX_ATTEMPTS} essais")


# generate_turn : testé avec de faux LLM, car google.colab.ai n'existe que sur Colab.
def fake_llm(prompt):
    return '```json\n{"user": "réponse générée", "feedback": "feedback généré"}\n```'

replies = ["Désolé, je ne peux pas.", '{"user": "", "feedback": "vide"}', '{"user": "ok", "feedback": "bien"}']

def flaky_llm(prompt):
    return replies.pop(0)

turn = generate_turn(fiches[0], first_question, "PARTIEL", fake_llm)
assert turn == {"user": "réponse générée", "verdict": "PARTIEL", "feedback": "feedback généré"}
assert build_dialogue(fiches[0], first_question, [turn])["messages"][2]["content"].startswith("[VERDICT: PARTIEL]\n")
assert generate_turn(fiches[0], first_question, "PARFAIT", flaky_llm)["user"] == "ok"
try:
    generate_turn(fiches[0], first_question, "INCORRECT", lambda prompt: "pas de JSON")
    assert False, "des sorties toujours invalides doivent lever une erreur"
except ValueError as error:
    print("generate_turn : erreur attendue →", error)

In [ ]:
def generate_case(concept: str, verdict: str, llm) -> dict:
    """Demande au LLM un cas complet sur un concept : mini-fiche, réponse d'apprenant et feedback.

    Paramètres :
        concept : nom du concept, choisi par nous (exemple : "Le paradoxe de Zénon").
        verdict : "PARFAIT", "PARTIEL" ou "INCORRECT".
        llm : fonction qui reçoit un prompt et renvoie un texte (sur Colab : ai.generate_text).

    Retour :
        dialogue {"messages": [system, user, assistant]} (format T4.1).

    Exemple :
        generate_case("Le paradoxe de Zénon", "PARTIEL", ai.generate_text)["messages"][0]["role"]  # "system"
    """
    if verdict not in VERDICTS:
        raise ValueError(f"verdict inconnu : {verdict}")
    prompt = (
        "Tu aides à créer des exemples pour entraîner un tuteur.\n\n"
        f"Concept : {concept}\n\n"
        "Écris dans l'ordre :\n"
        "1. une explication courte du concept (5 à 8 phrases) ;\n"
        "2. une question ouverte posée à l'apprenant sur ce concept ;\n"
        "3. 2 à 4 points clés attendus dans une bonne réponse ;\n"
        f"4. la réponse d'un apprenant qui mérite le verdict {verdict} ({VERDICT_RULES[verdict]}) ;\n"
        "5. le feedback du tuteur : ce qui est compris et ce qui manque, sans donner la réponse complète.\n\n"
        "Réponds uniquement avec un objet JSON de la forme "
        '{"explication": "...", "enonce": "...", "points_cles": ["...", "..."], "user": "...", "feedback": "..."}.'
    )
    for _ in range(MAX_ATTEMPTS):
        raw = llm(prompt)
        # les LLM entourent souvent le JSON de texte ou de ```json : on garde ce qui est entre accolades
        json_text = raw[raw.find("{") : raw.rfind("}") + 1]
        try:
            data = json.loads(json_text)
            explication = data["explication"].strip()
            enonce = data["enonce"].strip()
            user = data["user"].strip()
            feedback = data["feedback"].strip()
            points_cles = [point.strip() for point in data["points_cles"]]
        except (ValueError, KeyError, TypeError, AttributeError):
            continue  # sortie inutilisable : on redemande
        texts_ok = explication and enonce and user and feedback and "[VERDICT" not in feedback
        # 2 à 4 points clés, comme dans le contrat 1 ; une chaîne à la place de la liste serait lue lettre par lettre
        points_ok = isinstance(data["points_cles"], list) and 2 <= len(points_cles) <= 4 and all(points_cles)
        if texts_ok and points_ok:
            # même forme qu'une fiche du dossier json : le message système garde la même structure
            fiche = {"concept": concept, "explication": [{"contenu": explication}]}
            question = {"enonce": enonce, "points_cles": points_cles, "apres_section": 1}
            return build_dialogue(fiche, question, [{"user": user, "verdict": verdict, "feedback": feedback}])
    raise ValueError(f"{concept} ({verdict}) : aucune sortie valide après {MAX_ATTEMPTS} essais")


# generate_case : un cas complet inventé par un faux LLM donne un dialogue au même format.
fake_case = {
    "explication": "explication d'essai",
    "enonce": "question d'essai ?",
    "points_cles": ["point d'essai 1", "point d'essai 2"],
    "user": "réponse générée",
    "feedback": "feedback généré",
}
case = generate_case("concept d'essai", "PARTIEL", lambda prompt: json.dumps(fake_case))
system_content = case["messages"][0]["content"]
assert [message["role"] for message in case["messages"]] == ["system", "user", "assistant"]
assert system_content.startswith(SYSTEM_PROMPT) and "Concept : concept d'essai" in system_content
assert "question d'essai ?" in system_content and "- point d'essai 2" in system_content
assert case["messages"][2]["content"] == "[VERDICT: PARTIEL]\nfeedback généré"
try:
    one_point = dict(fake_case, points_cles=["un seul point"])
    generate_case("concept d'essai", "PARFAIT", lambda prompt: json.dumps(one_point))
    assert False, "un cas avec un seul point clé doit lever une erreur"
except ValueError as error:
    print("generate_case : erreur attendue →", error)

In [ ]:
# Sortie : un dialogue par question, avec un tour d'essai en attendant la vraie graine.
placeholder_seed = [
    {"question_id": question["id"], "turns": turns[:1]}
    for fiche in fiches
    for question in fiche["questions"]
]
lines_count = write_jsonl(build_dataset(fiches, placeholder_seed), OUTPUT_PATH)
print(f"{OUTPUT_PATH} : {lines_count} lignes écrites")

## Génération avec le LLM de Colab

Cette cellule ne tourne que sur la machine Colab (`google.colab.ai`). `call_llm` se passe en paramètre `llm` à `generate_turn` et `generate_case`.

In [ ]:
from google.colab import ai

# Modèle fixé : le modèle par défaut de Colab peut changer, et les réponses avec lui.
MODEL_NAME = "google/gemini-3.5-flash"


def call_llm(prompt: str) -> str:
    """Envoie un prompt au LLM de Colab et renvoie sa réponse.

    Paramètres :
        prompt : texte envoyé au modèle.

    Retour :
        texte généré par le modèle.

    Exemple :
        call_llm("Réponds uniquement par le mot : ok")  # "ok"
    """
    return ai.generate_text(prompt, model_name=MODEL_NAME)


# Test : le modèle fixé existe et répond par un texte non vide.
models = ai.list_models()
print("Modèles disponibles :", models)
assert MODEL_NAME in models, f"{MODEL_NAME} absent : choisir un modèle de la liste ci-dessus"
reply = call_llm("Réponds uniquement par le mot : ok")
assert isinstance(reply, str) and reply.strip()
print("call_llm :", reply.strip())